## Loading & tuning Redshift — COPY, Spectrum, dist/sort keys

The right way to bulk-load Redshift is the **`COPY`** command — reads files **in parallel** from S3 (each compute slice pulls its own), supports CSV/JSON/Parquet/ORC/Avro, and is orders of magnitude faster than `INSERT`. The near-universal pattern: **land data in S3, then `COPY` it in**. **Redshift Spectrum** is the other half — query data **directly in S3** without loading, via **external tables** (usually cataloged in the **Glue Data Catalog**), joining them against in-cluster tables; Spectrum spins up its own scan nodes, billed per TB scanned. The split: **hot, frequently-joined data inside Redshift; cold history in S3 via Spectrum**.

Two physical-design choices make or break performance on large tables. **Distribution style** decides how rows spread across nodes: **AUTO** (Redshift chooses — the default), **KEY** (hash a column so equal values co-locate — for large tables joined on that column), **ALL** (full copy on every node — for small dimension tables joined a lot), **EVEN** (round-robin — no clear join key). Classic pattern: big `orders` fact **KEY** on `customer_id`, small `customers` dim **ALL** → joins run **locally, no network shuffle**.

**Sort keys** decide storage order within a node; Redshift keeps **zone maps** (min/max per block) so a filter on the sort key **skips blocks** that can't match — massively cutting I/O for range-filtered and time-series queries.